Explore the **OntoSecAI** knowledge base with SPARQL (with rdflib).

OntoSecAI (*Ontology-Driven AI Security*) complements MITRE D3FEND: D3FEND covers
classic technical countermeasures, OntoSecAI covers **AI / agentic** ones. Its 79
mitigations come from MITRE ATLAS (`AML.M*`), the OWASP LLM Top 10 (`OWASP.M*`)
and CAPEC (`CAPEC.M*`).

Key structure of the ontology (useful to read the queries):
  * Everything lives in ONE namespace, `hes:`
    `http://www.semanticweb.org/ubaid/ontologies/2023/5/HES#`
  * A mitigation is an OWL **individual** of class `hes:Mitigations`, carrying
    `hes:Name`, `hes:Description` and `hes:IsIncludedIn` (an AI-lifecycle phase).
  * An attack is an individual carrying `hes:Name`, `hes:Description` and:
        `hes:IsMitigatedBy` -> the mitigations that address it
        `hes:Violates`      -> Confidentiality / Integrity / Availability
        `hes:Exploits`      -> a CWE or an OWASP-LLM weakness
        `hes:IsInitiatedBy` -> the threat actor
        `hes:Has`           -> a risk level
  * `hes:LeadsTo` links an ATLAS technique (`AML.T*`) to the attack it enables.

**IMPORTANT difference with D3FEND.** D3FEND is *artifact-centric*: a technique is
attached to a digital artifact by an OWL restriction carrying a verb, so you can
query `artifact -> techniques -> filter by verb`. OntoSecAI has **no artifact
taxonomy**: it is *attack-centric*, and its mitigations are a flat list of
individuals. So the retrieval chain is different -- you either go through an
ATTACK (`IsMitigatedBy`), or you match the mitigation catalogue directly.

In [1]:
from rdflib import Graph, Literal, URIRef

In [2]:
g = Graph().parse("AISecurityKnowledgeBase.rdf")   # RDF/XML, not turtle

In [3]:
P = """
PREFIX hes:  <http://www.semanticweb.org/ubaid/ontologies/2023/5/HES#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX owl:  <http://www.w3.org/2002/07/owl#>
"""

In [4]:
def run(title, q, limit=8):
    print(f"\n===== {title} =====")
    rows = list(g.query(P + q))
    for r in rows[:limit]:
        print("  " + " | ".join(
            "" if v is None else str(v).split("#")[-1] for v in r))
    print(f"  ({len(rows)} rows)")

#### Exploring the knowledge base

In [5]:
# Q0: which relations does the ontology actually use? A quick way to learn the
# schema of an unfamiliar KB before writing any real query.
run("Q0  relations in use", """
SELECT ?prop (COUNT(*) AS ?n) WHERE {
  ?s ?prop ?o .
  FILTER( STRSTARTS(STR(?prop),
          "http://www.semanticweb.org/ubaid/ontologies/2023/5/HES#") )
} GROUP BY ?prop ORDER BY DESC(?n)""", 12)


===== Q0  relations in use =====
  Description | 352
  Name | 250
  IsIncludedIn | 92
  Exploits | 90
  Has | 88
  IsMitigatedBy | 84
  Violates | 51
  LeadsTo | 50
  IsInitiatedBy | 42
  Examples | 12
  AttackScenarios | 10
  Values | 8
  (12 rows)


In [6]:
# Q1: the mitigation catalogue, grouped by its source framework.
# The id prefix (AML / OWASP / CAPEC) tells you where each entry comes from.
run("Q1  mitigations by source", """
SELECT ?source (COUNT(?m) AS ?n) WHERE {
  ?m a hes:Mitigations ; hes:Name ?name .
  BIND( STRBEFORE( STRAFTER(STR(?m), "#"), "." ) AS ?source )
} GROUP BY ?source ORDER BY DESC(?n)""")


===== Q1  mitigations by source =====
  AML | 40
  OWASP | 35
  CAPEC | 6
  (3 rows)


In [24]:
# Q2: where each mitigation sits in the AI lifecycle. This is OntoSecAI's
# organizing axis -- the closest thing it has to D3FEND's tactics.
run("Q2  mitigations per AI lifecycle phase", """
SELECT ?phase (COUNT(?m) AS ?n) WHERE {
  ?m a hes:Mitigations ; hes:IsIncludedIn ?phase .
} GROUP BY ?phase ORDER BY DESC(?n)""", 12)


===== Q2  mitigations per AI lifecycle phase =====
  ModelEvaluation | 14
  Monitoring&Maintenance | 13
  ModelEngineering | 13
  Deployment | 12
  HardwareSecurityEngineering | 9
  AccessManagement | 8
  DataPreparation | 8
  SoftwareSecurityEngineering | 7
  Business&DataUnderstanding | 6
  AdditionalActivities | 2
  (10 rows)


In [25]:
# Q3: the attacks that have at least one mitigation attached.
run("Q3  attack catalogue", """
SELECT ?id ?name WHERE {
  ?a hes:IsMitigatedBy ?m ; hes:Name ?name .
  BIND( STRAFTER(STR(?a), "#") AS ?id )
} GROUP BY ?id ?name ORDER BY ?id""", 10)


===== Q3  attack catalogue =====
  AML.A0001 | Detection Evasion
  AML.A0002 | Bypassing Detection
  AML.A0003 | Resource Hijacking
  AML.A0004 | Replication Attack
  AML.A0005 | Misconfiguration of Source file/Repository
  AML.A0006 | Chatbot Poisoning
  AML.A0007 | Denial of Services
  AML.A0008 | GPT Poisoning
  AML.A0009 | Authentication Evasion
  AML.A0010 | Backdoor Attack
  (33 rows)


In [26]:
# Q4: the full context of one attack. Note how much richer this is than a
# bare mitigation list: CIA property violated, weakness exploited (CWE + OWASP-LLM), threat actor, risk level.
run("Q4  full context of 'Prompt Injection'", """
SELECT ?aname ?prop ?target WHERE {
  ?a hes:Name "Prompt Injection" .
  ?a ?prop ?target .
  ?a hes:Name ?aname .
  FILTER( ?prop IN (hes:Violates, hes:Exploits, hes:IsInitiatedBy, hes:Has) )
} ORDER BY ?prop""", 12)


===== Q4  full context of 'Prompt Injection' =====
  Prompt Injection | Exploits | CWE-20
  Prompt Injection | Exploits | CWE-362
  Prompt Injection | Exploits | LLM01-PromptInjectionVulnerabiliy
  Prompt Injection | Has | MediumRisk
  Prompt Injection | IsInitiatedBy | Bots
  Prompt Injection | Violates | Integrity
  (6 rows)


#### Operationalization pipeline

Example leaf goal (ISO/IEC 42001 A.9.5): **"Monitor inputs and outputs of AI systems"**
-- in the Agentforce scenario, this is the goal that defends against a manipulated
conversation.

Chain (the OntoSecAI analogue of the D3FEND chain):
1) find the ATTACK the goal defends against
2) mitigations for that attack  (`IsMitigatedBy`)
3) filter by AI lifecycle phase
4) verify definitions

For another goal, change: the attack name, the phases, the ids.


In [27]:
# STEP 1: Find the attack the goal defends against.
# Keyword-search the attack names/descriptions with a term from the goal
# ("manipulated input", "injection" --> Prompt Injection).
# In D3FEND this step looks for a digital ARTIFACT; here it looks for an ATTACK,
# because OntoSecAI is attack-centric and has no artifact taxonomy.
run("STEP 1  find the attack ('prompt injection')", """
SELECT ?id ?name ?definition WHERE {
  ?a hes:Name ?name ; hes:Description ?definition .
  BIND( STRAFTER(STR(?a), "#") AS ?id )
  FILTER( CONTAINS(LCASE(?name), "injection") )
} ORDER BY ?id""", 5)


===== STEP 1  find the attack ('prompt injection') =====
  AML.A0013 | Prompt Injection | In AI systems, especially those dealing with natural language processing, prompt injection involves manipulating the system's output by injecting crafted inputs or prompts. This can mislead the AI to generate responses that could be harmful or unauthorized.
  AML.A0017 | Fault Injection Attacks | Systems are actively disrupted by faulty input data or physical environment disruptions.
  AML.T0051 | LLM Prompt Injection | An adversary may craft malicious prompts as inputs to an LLM that cause the LLM to act in unintended ways. These "prompt injections" are often designed to cause the model to ignore aspects of its original instructions and follow the adversary's instructions instead.
Prompt Injections can be an initial access vector to the LLM that provides the adversary with a foothold to carry out other steps in their operation. They may be designed to bypass defenses in the LLM, or allow the adv

In [29]:
# STEP 2: All mitigations attached to that attack (the core query).
# `IsMitigatedBy` is how OntoSecAI links an attack to its countermeasures --
# the counterpart of D3FEND's [ owl:onProperty ?verb ; owl:someValuesFrom ?artifact ].
# The optional phase tells you WHERE in the AI lifecycle the mitigation applies.
run("STEP 2  mitigations for Prompt Injection", """
SELECT ?a ?aname ?mid ?mname ?phase WHERE {
  ?a hes:Name ?aname ;
     hes:IsMitigatedBy ?m .
  ?m hes:Name ?mname .
  OPTIONAL { ?m hes:IsIncludedIn ?phase }
  BIND(STRAFTER(STR(?m), "#") AS ?mid)
  FILTER(?aname = "Prompt Injection")
}
ORDER BY ?mid""")


===== STEP 2  mitigations for Prompt Injection =====
  AML.A0013 | Prompt Injection | AML.M0003 | Model Hardening | ModelEngineering
  AML.A0013 | Prompt Injection | AML.M0015 | Adversarial Input Detection | ModelEngineering
  AML.A0013 | Prompt Injection | AML.M0015 | Adversarial Input Detection | ModelEvaluation
  AML.A0013 | Prompt Injection | OWASP.M0010 | Input validation | 
  AML.A0013 | Prompt Injection | OWASP.M0019 | Input Validation | DataPreparation
  AML.A0013 | Prompt Injection | OWASP.M0022 | User Confirmation | AccessManagement
  (6 rows)


In [30]:
# STEP 3: Filter by the lifecycle phase matching the goal's intent.
# A runtime governance goal wants mitigations that are enforceable at run time
# (AccessManagement, DataPreparation), not ones applied while training a model
# (ModelEngineering). This is the counterpart of D3FEND's verb filter.
run("STEP 3  filter by lifecycle phase (runtime-enforceable)", """
SELECT ?mid ?mname ?phase WHERE {
  ?a hes:Name "Prompt Injection" ; hes:IsMitigatedBy ?m .
  ?m hes:Name ?mname ; hes:IsIncludedIn ?phase .
  BIND( STRAFTER(STR(?m), "#") AS ?mid )
  FILTER( CONTAINS(STR(?phase), "AccessManagement")
       || CONTAINS(STR(?phase), "DataPreparation") )
} ORDER BY ?mid""")


===== STEP 3  filter by lifecycle phase (runtime-enforceable) =====
  OWASP.M0019 | Input Validation | DataPreparation
  OWASP.M0022 | User Confirmation | AccessManagement
  (2 rows)


In [13]:
# STEP 4: Verify the retained mitigations by their definition before attaching
# them as actions. List every retained id at once with IN(...).
run("STEP 4  verify definitions of retained mitigations", """
SELECT ?mid ?mname ?definition WHERE {
  ?m hes:Name ?mname ; hes:Description ?definition .
  BIND( STRAFTER(STR(?m), "#") AS ?mid )
  FILTER( ?mid IN ("OWASP.M0019", "OWASP.M0022") )
} ORDER BY ?mid""")


===== STEP 4  verify definitions of retained mitigations =====
  OWASP.M0019 | Input Validation | Filter malicious inputs to avoid model poisoning.
  OWASP.M0022 | User Confirmation | Require manual authorization for sensitive actions.
  (2 rows)


Leaf goal : Monitor inputs and outputs of AI systems

Actions :
- OWASP.M0019 Input Validation
- OWASP.M0022 User Confirmation